# Mixed Precision Training

> Section 21 covered inference-side quantization: compressing trained weights from FP16 to INT4 / INT8 to save memory and speed up decode. Training faces a different class of problems — gradients are several orders of magnitude smaller than weights, so using low precision directly would round the accumulation of an entire batch down to zero and kill training. This section covers training-side precision strategies: from FP16 loss scaling, to BF16's "just works" simplicity, to the 2024-2026 frontier of FP8 / FP4.
>
> The path runs in seven steps. We first open up the bit-level structure of floating point numbers to see how "precision" and "range" emerge from the exponent and mantissa bits; then we walk each precision format to see what happens in training and why problems arise; finally we land on a decision tree: which precision to use for dense models, MoE models, pretraining, and fine-tuning.


Training precision and inference precision both appear to "compress the bit count" on the surface, but their goals differ. Inference only reads weights and runs one forward pass — errors are static, and a bad quantization just produces slightly worse output. Training runs forward + backward + gradient accumulation thousands of times; each step's error is amplified by the next step's gradient. A seemingly tiny precision issue at step 5000 can suddenly turn into NaN.

So the core tension of training precision comes down to two statements: matrix multiplication is extremely tolerant of noise (matmul is an averaging effect; a small amount of rounding error gets diluted), but small-value accumulation is extremely sensitive to precision (adding 0.00001 to 1.0 in FP16 simply loses the 0.00001). This entire appendix searches for a balance around this pair of tensions — which ops can use low precision, which must stay FP32, and how to use a scaling factor to shift the effective range of low precision toward the small-value side.

First, let us look at a bit-level picture of "what a number looks like in memory".


## 1. Bit-level structure of floating point numbers

A floating point number is composed of three segments of bits: one sign bit, several exponent bits, and the remaining mantissa bits. Together they determine how large and how precise a number can be represented.

- Sign bit: 1 bit, 0 for positive and 1 for negative
- Exponent bits: determine the dynamic range — how many orders of magnitude apart the representable maximum and minimum are
- Mantissa bits: determine the precision — how close two numbers in the same order of magnitude can be while still being distinguished

More exponent bits means larger range; more mantissa bits means higher precision. When the total number of bits is fixed, the two compete for bits. This is the starting point of BF16 design: it is also 16 bit like FP16, but it gives up 3 mantissa bits to the exponent in exchange for a larger range and lower precision.


In [ ]:
# Lay out the bit allocation of common dtypes in a single table
import numpy as np
import torch
import pandas as pd
# Note: every cell in this appendix imports things locally, not at the top

dtypes_info = [
    # name, total_bits, sign, exponent, mantissa
    ('FP32',   32, 1, 8, 23),
    ('FP16',   16, 1, 5, 10),
    ('BF16',   16, 1, 8, 7),
    ('FP8 E4M3',  8, 1, 4, 3),
    ('FP8 E5M2',  8, 1, 5, 2),
    ('FP8 E3M4',  8, 1, 3, 4),  # rare variant, used in some research
    ('FP4 E2M1',  4, 1, 2, 1),
    ('INT8',    8, 0, 0, 8),   # integers have no exponent bits
    ('INT4',    4, 0, 0, 4),
]

print(f"{'dtype':<12}{'bits':>6}{'sign':>6}{'exp':>6}{'mant':>6}  {'note'}")
print('-' * 70)
for name, total, sign, exp, mant in dtypes_info:
    note = ''
    if name == 'FP32':
        note = 'baseline for master weight / optimizer state in training'
    elif name == 'FP16':
        note = 'small range (max~=65504), requires loss scaling'
    elif name == 'BF16':
        note = 'same range as FP32, only 7-bit precision'
    elif name.startswith('FP8 E4M3'):
        note = 'forward workhorse: higher precision'
    elif name.startswith('FP8 E5M2'):
        note = 'backward gradient workhorse: larger range'
    elif name.startswith('FP4'):
        note = 'experimental, not yet common in pretraining'
    elif name.startswith('INT'):
        note = 'integer: needs an external scale to represent floats'
    print(f'{name:<12}{total:>6}{sign:>6}{exp:>6}{mant:>6}  {note}')

print()
print('Key observation: FP16 and BF16 are both 16 bit, but their exponent/mantissa split is exactly opposite.')
print('The two FP8 variants (E4M3 / E5M2) make the same tradeoff — one favors precision, the other range.')


### 1.1 The dynamic range vs precision tradeoff

Exponent bits determine range: roughly speaking, the maximum representable value is approximately $2^{2^{(n-1)}}$, where $n$ is the number of exponent bits. FP16's 5 exponent bits give an upper bound of $2^{2^4} = 2^{16} \approx 65504$; BF16's 8 exponent bits give $2^{2^7} \approx 3.4 \times 10^{38}$, the same order of magnitude as FP32.

Mantissa bits determine precision: how close two adjacent representable numbers can be. FP16's 10-bit mantissa gives a relative precision of about $2^{-10} \approx 0.001$; BF16 only has 7 mantissa bits, so its relative precision is only $2^{-7} \approx 0.008$, eight times worse.

Practical implication: BF16 can represent a small number like 0.0001 (it falls within the range), but imprecisely; FP16 simply cannot represent small gradients in some scenarios (they underflow directly to 0). Gradients in training are often on the order of $10^{-4}$ to $10^{-7}$, so range matters more than precision — this is the fundamental reason the whole community shifted to BF16 after 2022.


In [ ]:
# Inspect the actual range and precision differences with torch dtypes
import torch

dtypes = [torch.float32, torch.float16, torch.bfloat16]

print('=== Maximum representable value (dynamic range upper bound) ===')
for dt in dtypes:
    # torch.finfo returns the numeric info for the dtype
    info = torch.finfo(dt)
    print(f'{str(dt):<20} max = {info.max:<25.6e}  min = {info.min:.6e}')

print()
print('=== Smallest positive normal number (underflow threshold; below this it becomes 0) ===')
for dt in dtypes:
    info = torch.finfo(dt)
    print(f'{str(dt):<20} tiny = {info.tiny:<25.6e}')

print()
print('=== Relative precision (smallest relative gap between two adjacent numbers) ===')
for dt in dtypes:
    info = torch.finfo(dt)
    print(f'{str(dt):<20} eps = {info.eps:<25.6e}  (about {info.eps:.4f})')

print()
print('Key observations:')
print('  FP32 vs BF16 max is nearly identical (both reach 3.4e+38) — same range.')
print('  FP16 max is only 65504 — slightly larger intermediate activations overflow to Inf.')
print('  BF16 eps is 8x larger than FP16 — worse precision, but adequate for training.')


### 1.2 Concrete numbers: what 0.1 becomes at different precisions

"Precision loss" sounds abstract; let us look at the actual stored value of 0.1 under different dtypes. 0.1 is an infinite repeating decimal in base 10, and also infinite in binary, so any finite-bit float can only store an approximation.


In [ ]:
# Store the same 0.1 under different precisions, then read it back, and inspect the error
import torch

x = 0.1
print(f'Original value (Python float, essentially FP64): {x:.20f}')
print()

for dt in [torch.float32, torch.float16, torch.bfloat16]:
    # cast there and back, simulating "store then read back"
    stored = torch.tensor(x, dtype=dt).item()
    err = abs(stored - x)
    print(f'{str(dt):<20} stored as {stored:.20f}  error {err:.2e}')

print()
print('Now a scenario more sensitive to training: a small gradient 0.0001')
x = 0.0001
print(f'Original value: {x:.10e}')
print()
for dt in [torch.float32, torch.float16, torch.bfloat16]:
    stored = torch.tensor(x, dtype=dt).item()
    err = abs(stored - x)
    rel_err = err / x if x != 0 else 0
    print(f'{str(dt):<20} stored as {stored:.10e}  relative error {rel_err:.2e}')

print()
print('Key observation: both FP16 and BF16 can still hold 0.0001 (no underflow), but with different precision.')
print('If the gradient shrinks one more notch to 1e-7, FP16 with tiny=6e-5 cannot hold it and rounds it to 0.')


In [ ]:
# Demonstrate FP16 underflow: small gradients accumulate to 0
import torch

# Simulate gradient accumulation: sum gradients of 8 micro-batches
# each gradient is on the order of 1e-7 (common in real training)
grad_per_step = torch.full((4,), 1e-7, dtype=torch.float32)

# FP32 accumulation
acc_fp32 = torch.zeros(4, dtype=torch.float32)
for _ in range(8):
    acc_fp32 += grad_per_step

# FP16 accumulation (if the gradient itself were stored in FP16)
acc_fp16 = torch.zeros(4, dtype=torch.float16)
grad_fp16 = grad_per_step.to(torch.float16)
print(f'Single gradient as FP16: {grad_fp16}')
print(f'  (note: 1e-7 < FP16 tiny=6e-5, so it rounds to 0 directly)')
for _ in range(8):
    acc_fp16 += grad_fp16

# BF16 accumulation
acc_bf16 = torch.zeros(4, dtype=torch.bfloat16)
grad_bf16 = grad_per_step.to(torch.bfloat16)
print(f'Single gradient as BF16: {grad_bf16}')
for _ in range(8):
    acc_bf16 += grad_bf16

print()
print(f'FP32 accumulation result: {acc_fp32}')
print(f'FP16 accumulation result: {acc_fp16}  <- all zeros, gradient vanished')
print(f'BF16 accumulation result: {acc_bf16}  <- also loses precision, but better than FP16')
print()
print('This is why accumulation in training (gradient accumulation, optimizer state) must stay FP32.')


## 2. FP16 training

FP16 was the earliest low-precision format to be used at scale for training. NVIDIA's 2017 mixed precision training paper (Micikevicius et al.) systematically explained its two core problems and the corresponding engineering solutions — loss scaling and FP32 master weights.

The first problem is gradient underflow. FP16's smallest positive normal number is $6 \times 10^{-5}$, but gradients in training are often on the order of $10^{-4}$ to $10^{-7}$. Storing gradients directly as FP16 turns a substantial fraction into 0, so the weights effectively never receive that signal.

The second problem is distorted weight updates. Adding a gradient $\Delta w = 0.0001$ to $w = 1.0$ in FP16: $1.0 + 0.0001$ rounds back to $1.0$ — the update is simply lost.


### 2.1 Loss scaling: temporarily shifting gradients into FP16's visible range

NVIDIA's solution is called loss scaling. The idea is direct: since gradients are too small and get swallowed by FP16's lower bound, multiply the loss by a large number (e.g. $2^{16}$) before backward so that the gradients produced by the chain rule are also amplified by $2^{16}$ and land within FP16's representable range. Before updating the weights, divide the gradients back by $2^{16}$.

In pseudocode:

```text
scaled_loss = loss * S
scaled_loss.backward()       # gradients are amplified by S, avoiding underflow
for p in params:
    p.grad = p.grad / S       # restore
    p.data -= lr * p.grad     # update with the restored gradient
```

How large should S be? Fixed scaling uses a constant value (commonly $2^{16}$); it is simple but fragile — if a step's gradient is unexpectedly large, the scaled value exceeds 65504 and becomes Inf. Dynamic scaling is more robust: after each backward, check whether the gradients contain Inf / NaN; if so, halve S and redo the step; if not, keep or double S.


In [ ]:
# Mini demonstration of the effect of loss scaling
import torch

torch.manual_seed(42)
# Simulate a small gradient (common in deep layers during real training)
grad_true = torch.randn(8) * 1e-5  # order of 1e-5
print(f'True gradient (FP32): {grad_true}')
print()

# No scaling, just cast directly to FP16
grad_fp16_noscale = grad_true.to(torch.float16)
print(f'Directly cast to FP16: {grad_fp16_noscale}')
print(f'  Number of non-zero elements: {(grad_fp16_noscale != 0).sum().item()} / 8')
print(f'  -> a substantial fraction of gradients were swallowed by underflow')

print()
# loss scaling: amplify by S=1024, cast to FP16, then divide back
S = 1024
grad_scaled = (grad_true * S).to(torch.float16)
grad_restored = grad_scaled.to(torch.float32) / S
print(f'loss scaling (S={S}):')
print(f'  Amplified then cast to FP16: {grad_scaled}')
print(f'  Divided back by S:          {grad_restored}')
print(f'  Max deviation from truth:   {(grad_true - grad_restored).abs().max().item():.2e}')
print()
print('Key observation: after scaling, all non-zero gradients are preserved and the update signal is not lost.')


In [ ]:
# Minimal implementation of dynamic loss scaling
import torch

class DynamicLossScaler:
    """
    Minimal dynamic loss scaler to illustrate the idea.
    The real implementation is torch.cuda.amp.GradScaler.
    """
    def __init__(self, init_scale=2**16, growth_factor=2.0, backoff_factor=0.5,
                 growth_interval=2000):
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self.steps_since_growth = 0

    def scale_loss(self, loss):
        return loss * self.scale

    def unscale_grads(self, grads):
        return [g / self.scale for g in grads]

    def step(self, grads_found_inf):
        """Call after each backward; grads_found_inf indicates whether Inf/NaN was detected"""
        if grads_found_inf:
            self.scale *= self.backoff_factor
            self.steps_since_growth = 0
            print(f'  Inf found, scale halved to {self.scale:.1f}')
        else:
            self.steps_since_growth += 1
            if self.steps_since_growth >= self.growth_interval:
                self.scale *= self.growth_factor
                self.steps_since_growth = 0
                print(f'  No Inf for a while, scale doubled to {self.scale:.1f}')


# Demo: simulate several training steps, occasionally injecting Inf
scaler = DynamicLossScaler(init_scale=1024)
inf_events = [False, False, True, False, False, False]  # inject Inf at step 3
for step, has_inf in enumerate(inf_events, 1):
    print(f'step {step}: scale={scaler.scale:.1f}', end='')
    scaler.step(has_inf)
print()
print('Key observation: dynamic scaling backs off automatically on Inf and grows slowly when there is no Inf,')
print('keeping the scale at the edge of "usable" — that is why it is more stable than fixed scaling.')


### 2.2 Which ops stay FP32

Not every op is suitable for FP16. NVIDIA's mixed precision training convention splits ops into two categories:

- **FP16-friendly**: matmul, conv. These ops consist of large-batch multiply-adds where rounding errors are averaged and diluted; FP16 is more than enough, and they can take advantage of the 2x speedup from Tensor Cores.
- **Must stay FP32**: reductions (sum, mean), softmax, layernorm, loss computation. These ops either involve cross-element accumulation (small errors get amplified) or involve exp / log (numerically range-sensitive), and tend to produce NaN in FP16.

PyTorch's `torch.autocast` dispatches according to this table — it casts matmul to FP16 automatically and keeps softmax / layernorm in FP32 automatically, so developers do not need to cast manually.


In [ ]:
# Demonstrate softmax's numerical instability under FP16
import torch

# A set of logits with a large maximum value
torch.manual_seed(0)
logits = torch.randn(4) * 20  # amplify the numeric range
print(f'logits: {logits}')

# FP32 softmax
probs_fp32 = torch.softmax(logits, dim=-1, dtype=torch.float32)
print(f'FP32 softmax: {probs_fp32}')

# FP16 softmax: first cast logits then run softmax (the wrong way)
logits_fp16 = logits.to(torch.float16)
probs_fp16_bad = torch.softmax(logits_fp16.float(), dim=-1)
print(f'FP16 softmax (wrong way): {probs_fp16_bad}')

# Compare the error
err = (probs_fp32 - probs_fp16_bad).abs().max().item()
print(f'Max error: {err:.2e}')
print()
print('Key observation: exp is sensitive to its input, and intermediate FP16 results easily overflow or lose precision.')
print('That is why PyTorch autocast keeps softmax in FP32 automatically and saves developers from this trap.')


### 2.3 Common pitfalls of FP16 training

Getting FP16 training running is easy, but so is stepping on a landmine. Three of the most common failure modes:

**NaN training crash.** A step where loss scaling fails to hold produces Inf gradients; the optimizer then updates weights with Inf, and every parameter in the model instantly becomes NaN, never to recover. The symptom is the loss suddenly becoming nan and the weight histogram being all NaN. The fix is for GradScaler to skip the update step when it detects Inf (skipping step).

**Optimizer state still FP32.** Adam's momentum and variance are long-running accumulators; after thousands of FP16 steps their precision is destroyed. So Adam's state must be FP32, even if the weights themselves are FP16.

**KQV proj distortion.** The $QK^T$ inside attention overflows FP16's range (attention scores can reach the hundreds when seq_len is large), so after softmax everything collapses to 0 or 1. So attention typically runs in FP32 internally, or uses a more numerically stable implementation like Flash Attention.


## 3. BF16 training

BF16 (Brain Float 16, designed by Google in 2017 for TPU) became the de facto standard for LLM training after 2022. Llama, Qwen, DeepSeek, and GPT-4 (presumed) all pretrain with BF16. This section explains why it displaced FP16.

The core difference is one sentence: BF16 and FP32 share exponent bits (both have 8), so their dynamic ranges are identical. The maximum value FP32 can represent, $3.4 \times 10^{38}$, BF16 can too; the smallest positive normal number FP32 can represent, $1.2 \times 10^{-38}$, BF16 can too. The cost is that the mantissa shrinks from 23 bits to 7, so precision is worse.

This tradeoff wins for training. The two main classes of values in training — weights (on the order of 0.01-1.0) and gradients (on the order of $10^{-4}$-$10^{-7}$) — both fall within BF16's range, so no loss scaling is needed. Worse precision is something matmul can tolerate, and softmax and layernorm still run in FP32 (handled by autocast). So the entire training loop simplifies to "cast to BF16, run forward, run backward, cast back to FP32 for the update".


### 3.1 torch.autocast per-op dtype decisions

PyTorch's `torch.autocast` context manager is the primary API for BF16 training. Inside an autocast scope, each op automatically selects its input dtype according to a whitelist:

```python
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    output = model(inputs)  # automatically picks BF16 or FP32 per op
```

The whitelist has two groups:

- **lower_precision_ops** (automatically cast to BF16): `Linear`, `Conv2d`, `matmul`, `bmm` and the rest of the matmul family
- **fp32_ops** (automatically kept in FP32): `softmax`, `layer_norm`, `cross_entropy`, `sum`, `mean`, `cumsum` and other ops that need numerical stability or do reductions

Developers do not need to cast manually; autocast inserts casts at each op's entry and exit automatically. The cost is some overhead from the casts inside forward, but it is negligible compared to the gain from matmul.


In [ ]:
# Demonstrate how autocast changes an op's actual dtype
# Both CPU and CUDA support autocast; pick device_type based on the environment
import torch
import torch.nn as nn

class MiniBlock(nn.Module):
    """A mini Transformer block: Linear -> softmax -> Linear"""
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, d)
        self.fc2 = nn.Linear(d, d)

    def forward(self, x):
        # fc1 is a matmul, runs as BF16 under autocast
        h = self.fc1(x)
        # softmax is automatically kept in FP32 under autocast (even when the outside is BF16)
        h = torch.softmax(h, dim=-1)
        # fc2 is a matmul again, back to BF16
        return self.fc2(h)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_type = 'cuda' if device == 'cuda' else 'cpu'
torch.manual_seed(42)
block = MiniBlock(64).to(device)
x = torch.randn(2, 8, 64, device=device)

# Without autocast: default FP32
print('=== Without autocast ===')
out = block(x)
print(f'fc1.weight dtype: {block.fc1.weight.dtype}')
print(f'output dtype:     {out.dtype}')

print()
# With BF16 autocast
print('=== With autocast(bfloat16) ===')
with torch.autocast(device_type=dev_type, dtype=torch.bfloat16):
    out = block(x)
print(f'fc1.weight dtype: {block.fc1.weight.dtype}  <- weights themselves are still FP32')
print(f'output dtype:     {out.dtype}  <- output is BF16')
print()
print('Key observation: autocast does not change the dtype of model weights (still FP32);')
print('it only feeds a cast copy of the weights to matmul ops during forward.')


### 3.2 Why the master weight is still FP32

This is the most easily overlooked detail in mixed precision training. Even when forward and backward both run in BF16, the "true body" of the model weights is still FP32. The flow is:

1. The model holds an FP32 master weight
2. Before each forward pass, a BF16 copy of the master weight is cast out for computation
3. The gradient obtained from backward is BF16
4. The gradient is cast back to FP32 and added to the master weight (FP32 accumulation)
5. The master weight subtracts lr * gradient to complete the update

Why not use BF16 as master directly? The reason is what we showed earlier: $w = 1.0$ plus $\Delta w = 0.0001$ is lost directly in BF16 (eps is 0.008). After tens of thousands of steps, all small gradient updates would be lost and the model would essentially never train.

PyTorch's `torch.cuda.amp` defaults to exactly this flow: model parameters are FP32, autocast casts copies during forward, and GradScaler casts gradients back to FP32 for accumulation during backward. The only thing the developer needs to do is "do not proactively cast the model to BF16 when calling `.to('cuda')`".


In [ ]:
# Demonstrate BF16 master weight accumulation vs FP32 master weight accumulation
import torch

# Simulate 1000 training steps, each with a gradient of 1e-4
n_steps = 1000
lr = 1e-4

# Option A: use BF16 directly as the master weight (wrong)
w_bf16 = torch.tensor([1.0], dtype=torch.bfloat16)
for _ in range(n_steps):
    grad = torch.tensor([lr], dtype=torch.bfloat16)
    w_bf16 = w_bf16 - grad  # BF16 accumulation

# Option B: FP32 master weight, BF16 only used for forward (correct)
w_fp32 = torch.tensor([1.0], dtype=torch.float32)
for _ in range(n_steps):
    grad = torch.tensor([lr], dtype=torch.float32)
    w_fp32 = w_fp32 - grad  # FP32 accumulation

print(f'Initial value: 1.0, train for {n_steps} steps, each step subtracts {lr}')
print(f'Theoretical final value: {1.0 - n_steps * lr:.4f}')
print(f'BF16 master weight: {w_bf16.item():.6f}')
print(f'FP32 master weight: {w_fp32.item():.6f}')
print()
print('Key observation: the BF16 master barely moved (1e-4 < BF16 eps=0.008, all updates lost).')
print('This is why the master weight must stay FP32.')


### 3.3 Hands-on: BF16 vs FP32 loss curve

Run 300 training steps on the nanoGPT in this repo and compare the FP32 and BF16 loss curves. The expected result is that the two curves essentially overlap — BF16's precision loss has almost no effect on the final outcome, which is exactly why it became mainstream.

If the reader's environment has no GPU, the code below will run on CPU, taking about 1-2 minutes.


In [ ]:
# BF16 vs FP32 training comparison (mini experiment, runs on both CPU and GPU)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

# Minimal char-level language model, using only Linear + Embedding + hand-written attention
# Does not depend on nn.TransformerEncoder (behavior varies across PyTorch versions)
class TinyCausalAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_k = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        b, t, d = x.shape
        qkv = self.qkv(x).view(b, t, 3, self.n_head, self.d_k)
        q, k, v = qkv.unbind(dim=2)  # each [b, t, n_head, d_k]
        q = q.transpose(1, 2)  # [b, n_head, t, d_k]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        # causal mask
        mask = torch.tril(torch.ones(t, t, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        h = attn @ v  # [b, n_head, t, d_k]
        h = h.transpose(1, 2).contiguous().view(b, t, d)
        return self.out(h)

class TinyBlock(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.attn = TinyCausalAttention(d_model, n_head)
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.fc2(F.gelu(self.fc1(self.ln2(x))))
        return x

class TinyLM(nn.Module):
    def __init__(self, vocab_size=64, d_model=64, n_layer=2, n_head=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(128, d_model)
        self.blocks = nn.ModuleList([TinyBlock(d_model, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, targets=None):
        b, t = x.shape
        pos = torch.arange(t, device=x.device)
        h = self.tok_emb(x) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            h = blk(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

torch.manual_seed(42)
vocab_size = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_type = 'cuda' if device == 'cuda' else 'cpu'
print(f'Running device: {device}')

# Random data (teaching demo; does not depend on shakespeare corpus download)
def get_batch(batch_size=16, seq_len=32):
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    return x, y

def train_one(dtype_label, n_steps=300):
    torch.manual_seed(42)
    model = TinyLM().to(device)
    # FP32 master weight is always kept; BF16 mode relies on autocast
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4)
    losses = []
    use_amp = dtype_label != 'fp32'
    amp_dtype = torch.bfloat16 if dtype_label == 'bf16' else torch.float16
    t0 = time.time()
    for step in range(n_steps):
        x, y = get_batch()
        optim.zero_grad()
        if use_amp:
            with torch.autocast(device_type=dev_type, dtype=amp_dtype):
                _, loss = model(x, y)
        else:
            _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        if step % 30 == 0 or step == n_steps - 1:
            losses.append((step, loss.item()))
    elapsed = time.time() - t0
    return losses, elapsed

print('Training FP32 ...')
losses_fp32, t_fp32 = train_one('fp32', n_steps=300)
print(f'  done in {t_fp32:.1f}s')
print('Training BF16 (autocast) ...')
losses_bf16, t_bf16 = train_one('bf16', n_steps=300)
print(f'  done in {t_bf16:.1f}s')

print()
print(f'{"step":<8}{"FP32 loss":<14}{"BF16 loss":<14}')
for (s, l1), (_, l2) in zip(losses_fp32, losses_bf16):
    print(f'{s:<8}{l1:<14.4f}{l2:<14.4f}')
print()
print(f'FP32 total time: {t_fp32:.1f}s, BF16 total time: {t_bf16:.1f}s')
print('Key observation: the BF16 and FP32 loss curves essentially overlap, but BF16 is usually faster (on GPU).')


In [ ]:
# Plot the loss curve comparison
import matplotlib.pyplot as plt

steps_fp32 = [s for s, _ in losses_fp32]
vals_fp32 = [l for _, l in losses_fp32]
steps_bf16 = [s for s, _ in losses_bf16]
vals_bf16 = [l for _, l in losses_bf16]

plt.figure(figsize=(8, 4.5))
plt.plot(steps_fp32, vals_fp32, marker='o', label='FP32 (master + compute)', linewidth=2)
plt.plot(steps_bf16, vals_bf16, marker='s', label='BF16 (autocast)', linewidth=2)
plt.xlabel('training step')
plt.ylabel('loss')
plt.title('BF16 vs FP32 training loss (random data)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Key observation: on modern Transformers, the BF16 loss curve usually overlaps almost exactly with FP32.')
print('This is why the whole community adopted BF16 as the default training precision after 2022.')


## 4. FP8 training

FP8 is the frontier of training precision in 2024-2026. The NVIDIA Hopper (H100) architecture natively supports FP8 Tensor Cores, and DeepSeek-V3 used FP8 to cut pretraining cost in half, pushing the whole community past the FP8 tipping point.

But FP8 training is far more complex than BF16. FP8 has only 8 bit, and whichever variant (E4M3 or E5M2) is used, both range and precision are much tighter than BF16. To make FP8 usable in training, **fine-grained quantization** is mandatory — instead of one scale per tensor, the tensor is split into small tiles with one scale each. This section explains the E4M3 / E5M2 division of labor, the principle of fine-grained scaling, and DeepSeek-V3's engineering implementation.


### 4.1 E4M3 (forward) vs E5M2 (backward gradients)

FP8 has two standard variants, each with its own role in training:

| Variant | Exponent bits | Mantissa bits | Range | Precision | Main use |
|:---|:---|:---|:---|:---|:---|
| E4M3 | 4 | 3 | $\pm 448$ | Higher | Forward (activations, weights) |
| E5M2 | 5 | 2 | $\pm 57344$ | Lower | Backward (gradients) |

Why E4M3 for forward and E5M2 for backward? Because forward activations have a relatively narrow numeric range but need precision (to distinguish different features), and E4M3's 3-bit mantissa fits better; backward gradients span several orders of magnitude but do not need to be as accurate (update direction is tolerant of noise), so E5M2's large range fits better.

NVIDIA H100's FP8 Tensor Core can process both variants in a single instruction, allowing a mixed scheme where forward uses E4M3 and backward uses E5M2 to land directly in practice.


In [ ]:
# Look at the ranges of the two FP8 variants (PyTorch 2.1+ supports float8_e4m3fn / e5m2)
import torch

if hasattr(torch, 'float8_e4m3fn'):
    fp8_dtypes = [
        ('FP8 E4M3', torch.float8_e4m3fn),
        ('FP8 E5M2', torch.float8_e5m2),
    ]
    print('=== Numeric info of the two FP8 variants ===')
    for name, dt in fp8_dtypes:
        info = torch.finfo(dt)
        print(f'{name}: max={info.max:<10.1f}  min={info.min:<10.1f}  '
              f'tiny={info.tiny:<10.2e}  eps={info.eps:.4f}')
else:
    print('Current PyTorch does not support FP8 dtypes (needs 2.1+), using a table instead:')
    print('  FP8 E4M3: max=448,    tiny=2^-6=0.0156, eps=0.125')
    print('  FP8 E5M2: max=57344,  tiny=2^-14=6e-5,  eps=0.25')

print()
print('=== Compare with BF16 / FP16 ===')
for dt in [torch.bfloat16, torch.float16]:
    info = torch.finfo(dt)
    print(f'{str(dt):<20}: max={info.max:<12.1f} tiny={info.tiny:<12.2e} eps={info.eps:.4f}')
print()
print('Key observation: FP8 range is much smaller than BF16 (448 vs 3.4e+38).')
print('This is the fundamental reason FP8 training needs fine-grained scaling —')
print('a single scale per tensor cannot cover the distribution of activation values.')


### 4.2 DeepSeek-V3's fine-grained quantization

The DeepSeek-V3 paper (2024) publicly documents their FP8 training scheme, whose core is **per-tile / per-block scaling factors**: activations and weights are split into small tiles, each with its own scale, anchoring the effective FP8 range onto the actual range of each tile.

Specifically, DeepSeek-V3 uses two-level scaling:

- **activation**: split into $1 \times 128$ tiles (one group per row of 128 elements), one scale per group
- **weights**: split into $128 \times 128$ blocks, one scale per group

Why so fine-grained? Because activations usually contain outliers — a few elements that are tens of times larger than the rest. With a single scale per tensor, the outliers push the scale large and all normal elements get crushed into a handful of FP8 levels. Per-tile scaling lets each tile's scale fit its own data distribution, so outliers affect only the tile they live in.

The cost is that the scaling factors themselves take extra memory (one FP32 scale per 128 elements) and the kernel implementation is much more complex — it has to handle the per-block scale dequantization during the FP8 matmul. This is exactly the engineering problem DeepGEMM solves.


In [ ]:
# Demonstrate the precision difference of per-tensor vs per-tile scaling
import torch
import numpy as np

torch.manual_seed(0)
# Simulate an activation with a few outliers
x = torch.randn(128) * 0.5
x[5] = 8.0   # outlier
x[60] = -7.0

# Simulate FP8 E4M3 quantization (max=448; we use a smaller dynamic range for the demo)
def quantize_dequantize(x, scale, n_levels=127):
    """Symmetric quantization + dequantization"""
    x_q = torch.round(x / scale).clamp(-n_levels, n_levels)
    return x_q * scale

# Option 1: per-tensor (one shared scale for the whole segment)
scale_per_tensor = x.abs().max().item() / 127
x_deq_per_tensor = quantize_dequantize(x, scale_per_tensor)

# Option 2: per-tile (split into 128 groups? here we use 8 groups for the demo, 16 elements each)
# Real DeepSeek-V3 uses 1x128; we use a smaller tile here to make the difference clearer
n_tiles = 8
tile_size = len(x) // n_tiles
x_deq_per_tile = torch.zeros_like(x)
for i in range(n_tiles):
    chunk = x[i*tile_size:(i+1)*tile_size]
    s = chunk.abs().max().item() / 127
    x_deq_per_tile[i*tile_size:(i+1)*tile_size] = quantize_dequantize(chunk, s)

# Compare errors
err_per_tensor = (x - x_deq_per_tensor).abs().mean().item()
err_per_tile = (x - x_deq_per_tile).abs().mean().item()

# Look at the error of normal elements, excluding outliers
mask = torch.ones(128, dtype=torch.bool)
mask[[5, 60]] = False
err_normal_tensor = (x[mask] - x_deq_per_tensor[mask]).abs().mean().item()
err_normal_tile = (x[mask] - x_deq_per_tile[mask]).abs().mean().item()

print(f'Average error across the whole segment:')
print(f'  per-tensor: {err_per_tensor:.4f}')
print(f'  per-tile (8 groups): {err_per_tile:.4f}')
print(f'Average error of normal elements excluding outliers:')
print(f'  per-tensor: {err_normal_tensor:.4f}  <- outliers widen the scale, big precision loss for normal elements')
print(f'  per-tile:   {err_normal_tile:.4f}  <- outliers affect only their own segment')
print()
print('Key observation: per-tile confines the damage of outliers to the tile they live in,')
print('and other tiles can still use the full precision of FP8. This is the core motivation of DeepSeek-V3 fine-grained scaling.')


### 4.3 DeepGEMM: the engineering of keeping accumulation in BF16

DeepSeek's open-source [DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) is the core kernel library for FP8 matmul. Its key designs:

- **FP8 input, BF16 accumulation.** The product of FP8 x FP8 is around 16 bit, but a single matmul accumulates thousands of products. FP32 accumulation is the most precise but slow; BF16 accumulation is precise enough and runs on Tensor Cores. DeepGEMM chose BF16 accumulation as the compromise.
- **per-block scale handled inside the kernel.** The scale factor is not stored separately and then dequantized; it is fused into the matmul kernel, multiplying the corresponding scale as each block is accumulated. This saves a round trip to memory.
- **JIT compilation.** Kernels are compiled at runtime with NVRTC, generating specialized code for each matrix shape.
- **TMA + warp specialization.** Uses Hopper's TMA (Tensor Memory Accelerator) for asynchronous memory loads, combined with warp specialization to overlap memory access with computation.

Practical benefit for training: DeepSeek-V3 used FP8 training, and compared to BF16 on 2048 H100 GPUs cut training cost by about 50% (paper data) with almost no loss in final model quality.


### 4.4 FP8 training empirical data

**DeepSeek-V3** (2024, MoE, 671B total params / 37B active):

- Training precision: FP8 forward + FP8 backward gradient, per-tile / per-block scaling
- Accumulation precision: BF16 (DeepGEMM)
- Master weight: FP32
- Training cost: 148k H100 hours / 1 trillion tokens, about 50% lower than the BF16 baseline
- Effect: gap of < 0.5% on multiple benchmarks compared to a BF16-trained control (validated at small model scale)

**Llama 3 series fine-tuning** (Meta 2024, H100 cluster):

- Pretraining still predominantly BF16; FP8 is mainly used in the fine-tuning stage
- Uses the NVIDIA TransformerEngine library (which wraps FP8 kernels)
- FP8 fine-tuning cuts memory usage by about 30-40% and boosts throughput by about 1.5x

**Qwen2.5 / Qwen3 series**: the official technical reports do not explicitly state whether FP8 pretraining was used; community reproductions mostly use a combination of BF16 + FP8 fine-tuning.

One caveat: FP8 training has hard hardware requirements — it must be NVIDIA Hopper (H100 / H200) or newer Blackwell (B200); A100 / V100 do not support it. So FP8 training is still concentrated in leading companies; the community mostly uses BF16 pretraining + FP8 / INT8 fine-tuning.


## 5. FP4 / INT4 training

4-bit training is still in the research and exploration phase, and has not become a production standard like BF16. But 2024-2025 already saw a few influential works bringing 4-bit into actual use — mainly in **fine-tuning** rather than pretraining scenarios.

This section covers three representative works: QLoRA (INT4 fine-tuning, now widely used), BitNet (1.58-bit pretraining research), and why FP4 / INT4 pretraining is not yet common.


### 5.1 QLoRA: NF4 + double quantization

QLoRA (Dettmers et al., 2023) is currently the most mature 4-bit training work, but strictly speaking it does **4-bit fine-tuning** — a pretrained large model is kept frozen in INT4, and only a small LoRA adapter (FP16) on top is trained. This makes it possible to fine-tune a 65B model on a single 24GB GPU.

QLoRA's three key techniques:

- **NF4 (Normal Float 4-bit).** The 16 quantization levels of standard INT4 are uniformly distributed $[-7, -6, ..., 0, ..., 6, 7]$. But neural network weights usually follow a normal distribution with mean 0 and variance 1 — most weights are near 0 and few are far from it. NF4 places the 16 levels at the quantiles of a normal distribution, denser near 0 and sparser far away, which fits the typical weight distribution better.
- **Double quantization.** Quantization itself stores scale factors (one FP32 scale per group). When group_size is small (e.g. 64), the number of scales is substantial. Double quantization quantizes these scales themselves once more (FP32 -> INT8) to save another level of memory.
- **Paged optimizer.** Adam state is placed in CPU memory, and NVIDIA unified memory is used to auto-page when GPU memory overflows, avoiding OOM.

QLoRA has been widely reproduced on Llama-2 / Llama-3 / Qwen series, with results close to full-precision fine-tuning. But it **only applies to fine-tuning** — pretraining an INT4 model from scratch is still unstable today.


In [ ]:
# Demonstrate the core idea of NF4: quantile-based quantization vs uniform quantization
import numpy as np
from scipy.stats import norm

np.random.seed(42)
# Simulate a normally distributed weight
w = np.random.randn(10000).astype(np.float32)

# INT4 symmetric quantization: 16 levels, uniformly distributed in [-7, 7]
def int4_quant(w):
    scale = np.max(np.abs(w)) / 7
    q = np.round(w / scale).clip(-7, 7)
    return q * scale

# NF4: 16 levels placed at the quantiles of a standard normal
# quantile = norm.ppf((i + 0.5) / 16) for i in 0..15, then normalize to [-1, 1]
nf4_levels = norm.ppf((np.arange(16) + 0.5) / 16)
nf4_levels = nf4_levels / np.max(np.abs(nf4_levels))  # normalize to [-1, 1]

def nf4_quant(w):
    scale = np.max(np.abs(w))  # NF4 assumes the weights are already normalized
    w_norm = w / scale
    # Find the nearest NF4 level for each weight
    q = np.zeros_like(w)
    for i, val in enumerate(w_norm):
        idx = np.argmin(np.abs(nf4_levels - val))
        q[i] = nf4_levels[idx]
    return q * scale

w_int4 = int4_quant(w)
w_nf4 = nf4_quant(w)

mae_int4 = np.mean(np.abs(w - w_int4))
mae_nf4 = np.mean(np.abs(w - w_nf4))

print(f'Weight distribution: normal(0, 1), 10000 samples')
print(f'INT4 uniform quantization MAE: {mae_int4:.4f}')
print(f'NF4 quantile quantization MAE: {mae_nf4:.4f}  <- usually lower')
print()
print('Key observation: NF4 clusters its 16 levels near 0 (where the normal density is high),')
print('which fits the typical weight distribution better. This is why QLoRA uses NF4 instead of INT4.')


### 5.2 BitNet: 1.58-bit pretraining research

BitNet (Microsoft, 2023-2024) takes a more aggressive path: weights are directly trinarized to $\{-1, 0, +1\}$, so each weight needs only $\log_2 3 \approx 1.58$ bit. BitNet b1.58 (the 2024 version) demonstrated the possibility of pretraining from scratch.

The core idea:

- Weights have only three values, so matmul degenerates into add/subtract (multiply by -1 means negate, multiply by 0 means ignore, multiply by +1 means leave unchanged)
- Activations still use FP8 or BF16 (not quantized)
- Training uses the straight-through estimator (STE) to bypass the "trinarization is non-differentiable" problem

BitNet's paper data is striking: at the same parameter count, a 1.58-bit model can match or even exceed an FP16 baseline while drastically reducing memory and energy. But as of the end of 2025, BitNet is still in the research phase — no large-scale commercial model has been pretrained at 1.58-bit. The main obstacles are training stability (early loss tends to diverge) and ecosystem support (specialized kernels are needed).


### 5.3 Why FP4 / INT4 pretraining is not yet common

QLoRA showed 4-bit fine-tuning works, and BitNet showed 1.58-bit pretraining works at the research level, but large-scale 4-bit pretraining still has no production case. Three core obstacles:

**Training stability problems.** Early in pretraining the weight distribution has not yet formed, so the relative impact of quantization error is large. In BF16 training, the first 1000 steps of the loss curve essentially overlap with FP32; in 4-bit training, the first 1000 steps easily see loss spikes (sudden jumps) or even divergence. A warm-up phase starting at high precision and gradually switching to 4-bit is needed, which raises engineering complexity.

**Insufficient kernel ecosystem.** BF16 has mature cuBLAS / cuDNN support, and FP8 has DeepGEMM / TransformerEngine. 4-bit training kernels currently rely mostly on BitNet's own work and a few community implementations; no general-purpose library has emerged. Native support in training frameworks (PyTorch / Megatron / DeepSpeed) is also limited.

**Limited benefit.** The bottleneck of pretraining is usually activation memory and optimizer state, neither of which is easy to quantize to 4-bit (activations are precision-sensitive, and optimizer state needs long-term accumulation). Compressing weights from BF16 to 4-bit only saves the memory of the model itself, which is usually < 30% of total training memory.

So the current landscape is: pretraining is predominantly BF16 (some leading companies use FP8); fine-tuning is QLoRA (INT4 weights + FP16 adapter); inference uses INT4 / INT8 quantization. FP4 / 1.58-bit pretraining remains a research topic.


## 6. Common pitfalls summary

This section consolidates the pitfalls scattered across the previous chapters into two groups: general pitfalls (encountered by both dense and MoE), and MoE-specific pitfalls. Each group is organized as "symptom -> cause -> debugging direction" to serve as a checklist during real training.


### 6.1 General pitfalls (dense + MoE)

**Pitfall 1: Gradient underflow to 0**

- Symptom: training loss decreases normally at first, then suddenly plateaus after a few hundred steps; the weight histogram shows all gradients are 0
- Cause: in FP16 training gradients < $6 \times 10^{-5}$ become 0 directly; in BF16 training some extremely small gradients are also lost
- Debug: print the gradient's min / max / non-zero ratio; add loss scaling for FP16; for BF16 check whether all ops are inside the autocast scope

**Pitfall 2: KQV proj distortion**

- Symptom: attention weights are all 0 or 1, the model attends to no token at all
- Cause: under FP16/BF16 the $QK^T$ attention score reaches the hundreds when seq_len is large, and softmax saturates afterwards
- Debug: check the max value of the attention score; use Flash Attention (more numerically stable); keep attention internal computations in FP32

**Pitfall 3: Optimizer state precision problems**

- Symptom: after tens of thousands of training steps the loss stops decreasing, or even rebounds
- Cause: Adam's momentum / variance accumulate long-term; thousands of FP16 / BF16 steps destroy their precision
- Debug: check the dtype of optimizer.state — it must be FP32; custom optimizer implementations must explicitly store state as FP32

**Pitfall 4: Gradient accumulation precision loss across micro-batches**

- Symptom: with gradient accumulation enabled (accum_steps > 1), the result is worse than directly training with the same batch_size
- Cause: each micro-batch's gradient is BF16; after accumulating N times the precision loss compounds
- Debug: keep the accumulation buffer explicitly in FP32; PyTorch's `optimizer.zero_grad(set_to_none=True)` paired with manual FP32 accumulation


### 6.2 MoE-specific pitfalls

MoE models have additional landmines on precision, because expert routing is itself a numerically sensitive operation.

**Pitfall 1: Expert routing gating must be FP32**

- Symptom: after training for a while, some experts are never routed to (dead experts), or the routing oscillates violently between a few experts
- Cause: gating logits pass through softmax to produce routing probabilities; small-value softmax in FP16 / BF16 lacks precision, and top-1 routing's argmax becomes unstable
- Debug: keep the gating network (usually a small Linear + softmax) explicitly in FP32, outside of autocast
- Practice: DeepSeek-V3 / Qwen MoE gating all run in FP32

**Pitfall 2: Aux-loss precision sensitivity**

- Symptom: the load balancing loss value oscillates violently, or has no effect at all
- Cause: aux loss usually involves counts of how often experts are selected; these statistics are themselves small-value accumulations, and precision is lost in BF16
- Debug: compute aux loss explicitly in FP32; accumulate the buffer that tracks expert frequency in FP32

**Pitfall 3: Inter-expert load imbalance amplified by precision**

- Symptom: after training for a while, a few experts handle > 80% of tokens while other experts are undertrained
- Cause: precision differences systematically bias some experts' gating logits larger, forming a positive feedback loop
- Debug: monitor each expert's token share; increase the aux loss weight; keep gating strictly FP32

**Pitfall 4: Different precision strategies for shared vs routed experts**

- Symptom: the model as a whole is fine, but the features learned by routed experts overlap with those of the shared expert
- Cause: DeepSeek-V3 / Qwen MoE both have a shared expert (not involved in routing); if the shared expert uses BF16 and routed experts use FP8, the precision asymmetry causes the shared expert to take on more computation
- Debug: keep the shared expert and routed experts at the same dtype; or give the shared expert higher precision (it carries denser computation)


## 7. Practical guide

Land the previous content into a decision table. Two dimensions: training stage (pretraining / fine-tuning / inference) and model structure (dense / MoE).


### 7.1 Decision tree: which precision for which scenario

```text
Pretraining (from scratch)
+-- dense model
|   +-- with H100 / H200: BF16 master + FP8 forward (TransformerEngine)
|   +-- only A100: BF16 (autocast)
+-- MoE model
    +-- leading-company scale: FP8 forward + FP8 backward (DeepGEMM)
    +-- community scale: BF16 (gating strictly FP32)

Fine-tuning
+-- full-parameter fine-tuning: BF16 (same as pretraining)
+-- LoRA: base model BF16, adapter BF16
+-- QLoRA: base model INT4 (NF4), adapter FP16

Inference
+-- dense model: weight INT4 / INT8, activation FP16
+-- MoE model: weight INT4, activation FP16, gating FP32
```

Selection principle: pretraining prioritizes stability (BF16 / FP8), fine-tuning prioritizes memory (QLoRA), inference prioritizes throughput (INT4). For MoE models, gating must always be kept FP32 at every stage.


### 7.2 Checking whether training actually saved memory

Many people enable BF16 autocast but see no memory reduction, usually because the master weight is still FP32 + optimizer state is still FP32. The table below lists the actual memory savings of BF16 / FP8 training relative to FP32:

| Component | FP32 | BF16 autocast | FP8 (H100) |
|:---|:---|:---|:---|
| master weight | FP32 (4B/param) | FP32 (4B) | FP32 (4B) |
| forward weight copy | not needed | BF16 (2B, temporary) | FP8 (1B, temporary) |
| activation | FP32 | BF16 (2B) | FP8 (1B)|
| gradient | FP32 | BF16 -> FP32 (accumulate) | FP8 -> BF16 (accumulate) |
| Adam state | FP32 x 2 (8B) | FP32 x 2 (8B) | FP32 x 2 (8B) |

Actual benefit: BF16 saves about 30-40% total memory compared to FP32 (mainly from activation); FP8 saves another 20-30% compared to BF16 (further activation compression). Master weight and optimizer state are still FP32 and cannot be compressed.


In [ ]:
# Hands-on: compare the memory estimates of "full FP32 training" vs "BF16 autocast training"
import torch
import torch.nn as nn

def estimate_memory(model, mode='fp32'):
    """
    Estimate memory usage under two training modes.
    mode='fp32': full FP32 training (weight/grad/adam/activation all FP32)
    mode='bf16': BF16 autocast (master/grad/adam still FP32, activation is BF16)
    """
    n = sum(p.numel() for p in model.parameters())

    # master weight: FP32 in both modes
    master_weight = n * 4

    # gradient: accumulated to FP32 in both modes
    gradient = n * 4

    # Adam state: FP32 in both modes (momentum + variance)
    adam_state = n * 8

    # activation assumption: roughly equal to model parameter count (teaching simplification)
    if mode == 'fp32':
        activation = n * 4   # FP32 activation
    else:  # bf16
        activation = n * 2   # BF16 activation

    total = master_weight + gradient + adam_state + activation

    label = 'Full FP32 training' if mode == 'fp32' else 'BF16 autocast'
    print(f'\n=== {label} ({n/1e6:.1f}M params) ===')
    print(f'master weight (FP32):  {master_weight/1e9:.3f} GB')
    print(f'gradient (FP32):       {gradient/1e9:.3f} GB')
    print(f'Adam state (FP32x2):   {adam_state/1e9:.3f} GB')
    print(f'activation:            {activation/1e9:.3f} GB')
    print(f'total:                 {total/1e9:.3f} GB')
    return total

# Estimate with a 100M parameter model
class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1000, 100000)

model = DummyModel()
n = sum(p.numel() for p in model.parameters())
print(f'Model parameter count: {n/1e6:.1f}M')

t_fp32 = estimate_memory(model, 'fp32')
t_bf16 = estimate_memory(model, 'bf16')

saved = (1 - t_bf16 / t_fp32) * 100
print(f'\nBF16 saves {saved:.1f}% compared to full FP32')
print('Key observation: master weight + gradient + Adam state together take the large fixed chunk and cannot be compressed,')
print('so the memory benefit of low-precision training comes mainly from activation.')


### 7.3 Checklist for debugging NaN

Training crashing to NaN is the most common precision accident. Debug in this order:

1. **Check whether loss scaling is skipping steps.** During FP16 training, GradScaler should occasionally print "Skipping step" — if it skips dozens of steps in a row, the scale is too large or the model itself has a problem.
2. **Check gradient ranges.** After backward, print the min / max / mean of each parameter's gradient. If some parameter's gradient is Inf or NaN, locate the specific layer.
3. **Check attention scores.** If the max of $QK^T$ exceeds 1000, softmax saturates into one-hot and the gradient explodes during backward.
4. **Check LayerNorm variance.** If some hidden state has extremely small variance (< $10^{-6}$), LayerNorm's divide-by-variance makes the value explode.
5. **Check learning rate warmup.** Too large an lr in the first few hundred steps directly pushes weights to Inf.
6. **Check the data.** If a batch's token ID falls outside vocab_size, the Embedding lookup produces NaN.

Debugging tool: `torch.autograd.set_detect_anomaly(True)` prints the forward op that caused the NaN during backward, at the cost of slowing training by 2-3x.


## Appendix: FP8 engineering deep end

The previous sections covered the principles and paper data of FP8 training. This section is an engineering guide for readers who want to actually reproduce it: how to read the DeepGEMM source code, how to set up an FP8 fine-tuning environment from scratch.


### A3.1 DeepGEMM source reading guide

[DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) is DeepSeek's open-source FP8 GEMM library, one of the core components of DeepSeek-V3 training. The codebase is not large (about 2000 lines for the core), but reading it requires a CUDA + Hopper architecture background. Below is a guide organized by reading order.

**First pass: understand the API layer**

Read in this order:

1. `deep_gemm/__init__.py` — the exported public API: `fp8_gemm`, `grouped_fp8_gemm`, etc.
2. `deep_gemm/core/__init__.py` — the core data structures: the FP8 tensor wrapper, containing data and scale factor
3. `tests/test_fp8_gemm.py` — the unit tests, which make it clear how the API is used and what inputs/outputs look like

After this pass, you should be able to answer: is the FP8 tensor's scale factor per-tensor or per-block? What is the interface difference between grouped GEMM (the variant used by MoE) and regular GEMM?

**Second pass: understand the JIT compilation flow**

4. `deep_gemm/jit_kernels.py` — the JIT entry point; see how NVRTC compiles the CUDA kernel at runtime
5. The `.cuh` / `.cu` files under `csrc/` — the actual CUDA code; focus on `fp8_gemm_*.cuh`

During this pass, look closely at three things:

- **How per-block scale factors are stored.** DeepGEMM uses a separate scale tensor shaped to match the FP8 data. For example, if the weight is an FP8 tensor of shape `[K, N]`, the scale is an FP32 tensor of shape `[K//128, N//128]` (one scale per $128 \times 128$ block).
- **How BF16 accumulation is implemented.** The kernel's accumulator registers use BF16, but periodically upgrade to FP32 for a precise accumulation to avoid accumulation error getting too large. Look at the accumulator-related code under `csrc/include/deep_gemm/`.
- **The grid/block design of the kernel launch.** Each thread block processes a small output tile (e.g. $128 \times 128$); grid dimensions are partitioned by the output shape. TMA handles asynchronous loading of input tiles to shared memory, and warp specialization assigns "load" and "compute" to different warps to run in parallel.

**Third pass: understand grouped GEMM (MoE-specific)**

6. `deep_gemm/grouped_gemm.py` — the grouped GEMM interface for MoE scenarios
7. The corresponding CUDA kernel — see how it handles variable-length experts (variable-length grouped GEMM)

The problem grouped GEMM solves: in MoE each token is routed to different experts for a small matmul. Launching a separate GEMM kernel per expert has huge overhead. Grouped GEMM packs all experts' matmuls into a single kernel launch, processing them grouped by expert id.

**Required background**

Before reading DeepGEMM source, it is recommended to have:

- CUDA programming basics (grid / block / thread / shared memory / warp)
- Familiarity with NVIDIA Hopper architecture's TMA (Tensor Memory Accelerator) and warp specialization concepts
- Basic understanding of CUTLASS (NVIDIA's matrix multiplication template library); DeepGEMM borrows from CUTLASS's design


### A3.2 FP8 training reproduction guide

If you want to reproduce an FP8 training run from scratch (not read the source but actually run it), Llama-3 H100 FP8 fine-tuning is the recommended target — DeepSeek-V3 FP8 pretraining needs 2048 H100 GPUs and is not reproducible by the community; Llama-3 + H100 runs on a single card or 8 cards.

**Environment setup checklist**

```text
Hardware: NVIDIA H100 (80GB HBM3); a single card can fine-tune 7B, 8 cards can fine-tune 70B
CUDA: 12.x (H100 requires CUDA 12+)
cuDNN: 9.x (FP8 support)
PyTorch: 2.4+ (native FP8 dtype support)
Python: 3.10+
```

**Key dependency installation**

```bash
# TransformerEngine: NVIDIA's official FP8 training library, wrapping FP8 kernels
pip install transformer_engine[pytorch]

# Flash Attention 2 (paired with FP8 attention)
pip install flash-attn --no-build-isolation

# Training framework (pick one)
pip install trl          # HuggingFace's fine-tuning library, supports FP8
# or
pip install megatron-core # NVIDIA's large-model training framework
```

**Key hyperparameters and configuration**

A typical configuration for Llama-3 7B FP8 fine-tuning:

```python
import transformer_engine.pytorch as te
import torch

# Replace the model's nn.Linear with te.Linear (automatic FP8)
model = te.Linear(4096, 4096, fp8=True)

# Or use the te.fp8_autocast context during forward
with te.fp8_autocast(enabled=True):
    output = model(input)
```

Key hyperparameters:

- `fp8_margin`: safety margin for the scaling factor; usually 0-3; larger means more conservative
- `fp8_interval`: how many steps between recomputing the scaling factor; usually 1 (every step)
- `amax_history_len`: how many historical steps of amax are used to compute the scale; usually 1-16
- `amax_compute_algo`: scale computation algorithm, `max` (take the maximum) or `most_recent` (use the latest value)

**Verify that training is actually using FP8**

This is the easiest place to slip up — having the config in place does not mean FP8 is actually running. Two verification methods:

```python
# Method 1: print the actual dtype during forward
hook = model.register_forward_hook(
    lambda m, i, o: print(f'{m.__class__.__name__} output dtype: {o.dtype}')
)

# Method 2: use NVIDIA Nsight Systems profiler
# Run on the command line:
#   nsys profile -t cuda,nvtx python train.py
# Then in the Nsight Systems GUI check:
#   - whether there are cublasH8Gemm (FP8 GEMM) calls
#   - the fraction of total compute time taken by FP8 GEMM
```

If the profiler shows only `cublasGemmEx` (BF16 / FP16 GEMM) and no `cublasH8Gemm`, FP8 has not actually taken effect. Common causes: some ops are not inside the `fp8_autocast` scope, some layer of the model does not support FP8 and was fallen back, or the TransformerEngine version does not match the PyTorch version.


## Summary

- [ ] Can write out the exponent + mantissa bit allocation of FP32 / FP16 / BF16 / FP8 (E4M3 / E5M2) and explain the range vs precision tradeoff
- [ ] Understand why BF16 replaced FP16 as the training mainstream: same range, no loss scaling needed
- [ ] Can clearly explain why the master weight must be FP32: small-gradient accumulation is lost in BF16
- [ ] Understand `torch.autocast`'s per-op dtype decisions: matmul goes BF16, softmax / layernorm stay FP32
- [ ] Can explain FP8 training's E4M3 (forward) / E5M2 (backward) division of labor, and why per-tile / per-block scaling is necessary
- [ ] Know DeepSeek-V3 used FP8 to cut training cost by about 50%, and DeepGEMM's "FP8 input + BF16 accumulation" design
- [ ] Understand QLoRA's three techniques: NF4 + double quantization + paged optimizer
- [ ] Know that MoE models must keep gating in FP32, and why aux loss is precision-sensitive
- [ ] Can use the decision tree to pick a training precision for different scenarios (pretraining / fine-tuning / inference, dense / MoE)


## Exercises

The three exercises below help you ground the core concepts. You can ask an AI to help break down the steps or check your thinking, but it is not advisable to let the AI write the full answer directly — precision details only stick after you compute them yourself once.

**Exercise 1: Verify BF16 master weight accumulation**

Simulate what happens when accumulating a BF16 master weight for 1000 steps. Given the initial weight $w = 1.0$ and a per-step gradient $g = 0.0001$, do 1000 accumulation steps $w = w - g$ in BF16 and FP32 respectively, and compare the final values with the theoretical value.

Hint: BF16's eps is about 0.008; 0.0001 is far below this value, so accumulation loses precision.


In [ ]:
# Exercise 1: BF16 vs FP32 master weight accumulation
import torch

n_steps = 1000
grad = 1e-4

# TODO: accumulate in BF16 for n_steps steps
w_bf16 = None  # Hint: torch.tensor([1.0], dtype=torch.bfloat16), loop subtracting grad

# TODO: accumulate in FP32 for n_steps steps
w_fp32 = None

# TODO: compute the theoretical final value
w_theoretical = None  # 1.0 - n_steps * grad

assert w_bf16 is not None and w_fp32 is not None and w_theoretical is not None
assert abs(w_fp32.item() - w_theoretical) < 1e-4, f'FP32 should be close to the theoretical value, diff {abs(w_fp32.item() - w_theoretical)}'
print(f'Theoretical final value:  {w_theoretical:.6f}')
print(f'BF16 accumulation result: {w_bf16.item():.6f}')
print(f'FP32 accumulation result: {w_fp32.item():.6f}')
print(f'BF16 deviation from theory: {abs(w_bf16.item() - w_theoretical):.6f}')
print(chr(10004) + ' Exercise 1 passed: this is why the master weight must be FP32')


**Exercise 2: FP8 per-tile scaling error comparison**

Given a piece of data with outliers (128 elements, 3 outliers), quantize it with 4-bit symmetric quantization using per-tensor and per-tile (tile_size=16) respectively, and compute the average error of normal elements excluding outliers.

Hint: the key to per-tile is to first `reshape` into `[n_tiles, tile_size]`, compute a scale for each row independently, then quantize.


In [ ]:
# Exercise 2: per-tensor vs per-tile scaling
import torch

torch.manual_seed(42)
x = torch.randn(128) * 0.5
x[5], x[60], x[100] = 8.0, -7.0, 6.5  # 3 outliers

def quantize_dequantize(x, scale, n_levels=7):  # 4-bit: levels = 7
    x_q = torch.round(x / scale).clamp(-n_levels, n_levels)
    return x_q * scale

# TODO: per-tensor quantization (one shared scale for the whole segment)
scale_tensor = None  # x.abs().max() / 7
x_deq_tensor = None  # quantize_dequantize(x, scale_tensor)

# TODO: per-tile quantization (tile_size=16, 8 tiles in total)
tile_size = 16
x_deq_tile = torch.zeros_like(x)
# Hint: loop 8 times, each time processing the segment x[i*16:(i+1)*16]

assert scale_tensor is not None and x_deq_tensor is not None
mask = torch.ones(128, dtype=torch.bool)
mask[[5, 60, 100]] = False
err_tensor = (x[mask] - x_deq_tensor[mask]).abs().mean().item()
err_tile = (x[mask] - x_deq_tile[mask]).abs().mean().item()

print(f'per-tensor normal-element MAE: {err_tensor:.4f}')
print(f'per-tile  normal-element MAE: {err_tile:.4f}')
assert err_tile < err_tensor, 'per-tile should make the error of normal elements smaller (outliers affect only their own segment)'
print(chr(10004) + ' Exercise 2 passed: per-tile scaling is the key engineering trick for FP8 training precision')


**Exercise 3: Estimate how much memory BF16 training saves over full FP32 training**

Compare "full FP32 training" (weight + gradient + Adam state + activation all FP32) against "BF16 autocast training" (master weight + gradient + Adam state still FP32, only activation stored in BF16). Given a 1B-parameter model, assume the activation total is equivalent to 1B parameters (teaching simplification).

Hint: in full FP32 training every component is 4B or 8B per parameter; in BF16 autocast the master weight (4B) + gradient (4B) + Adam state (8B) stay the same, but activation drops from 4B/element to 2B/element.


In [ ]:
# Exercise 3: estimate training memory of a 1B model
# Convention: here we compare "full FP32 training" vs "BF16 autocast training" vs "FP8 training"
# Full FP32 training: weight, gradient, Adam state all FP32
# BF16/FP8 training: master weight + gradient + Adam state still FP32, only activation is saved

n_params = 1_000_000_000  # 1B

# Scenario A: full FP32 training (weight + gradient + adam all FP32)
# Hint: weight/gradient each 4B/param, Adam state (momentum+variance) 8B/param
weight_fp32_bytes = None       # 4B/param
gradient_fp32_bytes = None     # 4B/param
adam_state_bytes = None        # 8B/param

# Scenario B: BF16 autocast training
# master weight is still FP32, gradient accumulates to FP32, Adam state still FP32
# Only difference: activation is stored as BF16 (2B/element) instead of FP32 (4B/element) during forward
# Assume activation total is equivalent to 1B params (teaching simplification)
activation_fp32_bytes = None   # 4B x 1B (FP32 activation)
activation_bf16_bytes = None   # 2B x 1B (BF16 activation)

assert all(v is not None for v in [
    weight_fp32_bytes, gradient_fp32_bytes, adam_state_bytes,
    activation_fp32_bytes, activation_bf16_bytes
])

# Full FP32 training memory: weight + grad + adam + activation all FP32
total_fp32 = weight_fp32_bytes + gradient_fp32_bytes + adam_state_bytes + activation_fp32_bytes
# BF16 autocast: master FP32 + grad FP32 + adam FP32 + activation BF16
total_bf16 = weight_fp32_bytes + gradient_fp32_bytes + adam_state_bytes + activation_bf16_bytes

print(f'1B model training memory estimate (including activation):')
print(f'  Full FP32 training:        {total_fp32/1e9:.2f} GB')
print(f'  BF16 autocast training:    {total_bf16/1e9:.2f} GB (saves {(1-total_bf16/total_fp32)*100:.1f}%)')

assert total_bf16 < total_fp32, 'BF16 should save activation memory'
print(chr(10004) + ' Exercise 3 passed: master weight and optimizer state cannot be compressed, so the benefit of low precision comes mainly from activation')


## References

- Micikevicius et al., [Mixed Precision Training](https://arxiv.org/abs/1710.03740), 2017 — NVIDIA's foundational FP16 mixed precision training paper, which introduced loss scaling
- Kalamkar et al., [A Study of BF16 for Deep Learning](https://arxiv.org/abs/1905.12322), 2019 — a systematic study of BF16
- DeepSeek-AI, [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437), 2024 — the FP8 training fine-grained quantization scheme and training cost data
- DeepSeek-AI, [DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) — open-source FP8 GEMM library, a core component of DeepSeek-V3 training
- NVIDIA, [Transformer Engine](https://github.com/NVIDIA/TransformerEngine) — NVIDIA's official FP8 training library
- Micikevicius et al., [FP8 Formats for Deep Learning](https://arxiv.org/abs/2209.05433), 2022 — the FP8 E4M3 / E5M2 standardization paper
- Dettmers et al., [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314), 2023 — NF4 + double quantization + paged optimizer
- Wang et al., [BitNet: Scaling 1-bit Transformers for Large Language Models](https://arxiv.org/abs/2310.11453), 2023 — 1.58-bit pretraining research
